# Canonical Fast Tour — the whole myflopy surface in about a minute

The test suite shares one **contract-complete canonical model**: every one of
the 13 MF6 packages (DISV/NPF/STO/CHD/GHB/RCH/WEL/DRN/LAK/SFR/MVR/UZF/OC) and
all five observation-target families, validated by `CANONICAL_MODEL_CONTRACT`.
`CanonicalModelConfig.testing()` is the smallest grid that still satisfies
every contract clause — it builds and runs in **under a second**, so you can
rebuild it as often as you like and explore everything interactively.

This notebook: build + run the fast canonical model → (optionally) run the
entire myflopy test suite right here → explore heads, package results,
observation scatterplots, animations, and a real PESTPP-IES ensemble — all
through the preferred API. For the full-size story see `canonical_00` …
`canonical_06`.


In [ ]:
import sys
import time
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from IPython.display import display

import myflopy as mf
from canonical_notebook_style import notebook_header

notebook_header('FT', 'Canonical Fast Tour',
                'The smallest contract-complete canonical model: build, run, and explore everything — fast.')

root = Path('../artifacts/canonical_fast_tour')

config = mf.CanonicalModelConfig.testing()   # 21x21 grid, 4 layers, 6 periods
t0 = time.time()
model = mf.build_canonical_model(root / 'gwf', config=config)
t_build = time.time() - t0

t0 = time.time()
success, report = model.run_simulation()
t_run = time.time() - t0
assert success, report[-15:]

mf.CANONICAL_MODEL_CONTRACT.validate(model)   # every package + target family present
print(f'built in {t_build:.2f}s, mf6 ran in {t_run:.2f}s '
      f'({model.vor.ncpl} cells/layer x {model.gwf.modelgrid.nlay} layers, {model.nper} periods)')


## Run the whole myflopy test suite from here (optional)

The same `testing()` profile powers the test suite, so the **entire**
633-test suite — real mf6, PEST++, and MPI runs included — is quick enough to
fire from a notebook cell. Flip `RUN_SUITE = True` to prove the installation
end-to-end (`-n auto` uses pytest-xdist across your cores; drop it for a
serial run).


In [ ]:
RUN_SUITE = False
if RUN_SUITE:
    import subprocess

    repo_root = Path.cwd().parents[2]
    proc = subprocess.run(
        [sys.executable, '-m', 'pytest', '-q', '-n', 'auto'],
        cwd=repo_root,
        capture_output=True,
        text=True,
    )
    print(proc.stdout[-2500:])
    assert proc.returncode == 0
else:
    print('Set RUN_SUITE = True to run the full myflopy test suite right here.')


## Heads: dataframes, maps

`model.hds` is the heads explorer; `model.targets` holds the observation
families the canonical model registers at build time.


In [ ]:
# Simulated heads at the named observation wells, one column per well:
display(model.targets.heads.simulated_heads())

# Interactive head map (sectioned hover shows layers + surfaces per cell):
model.hds.map().plot().show()


In [ ]:
import matplotlib.pyplot as plt

from myflopy.modflow.mf6.grid.plotting import build_choropleth

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
roles = ['L1 upper unconfined', 'L2 lower unconfined', 'L3 aquitard', 'L4 confined']
for layer, (ax, role) in enumerate(zip(axes.ravel(), roles)):
    build_choropleth(model.vor, custom_zs=list(model.hds.array(layer=layer)),
                     layer=layer).plot_mpl(ax=ax, title=role)
fig.suptitle('Head by layer (final period)', fontsize=13)
fig.tight_layout()


## Animations

The unified grammar's `animate` verb works on any explorer leaf — maps over
stress periods, cross sections along a line, or series plots.


In [ ]:
# Head map animated across all stress periods:
model.hds.animate(kind='map', layer=0).show()


In [ ]:
from shapely.geometry import LineString

# A west-east cross-section line through the valley center:
xmin, ymin, xmax, ymax = model.vor.gdf_vorPolys.total_bounds
line = LineString([(xmin, (ymin + ymax) / 2), (xmax, (ymin + ymax) / 2)])
model.hds.animate(kind='xs', line=line).show(renderer='browser')


## Package results: stream profile, lake budget, exchange maps

Every package explorer exposes the same verbs — dataframes via `.get()` /
purpose-built tables, `map`, `plot`, `mosaic`, `animate`.


In [ ]:
per = model.nper - 1
model.packages.sfr.results.profile.plot(per=per, plot_fig=True)

In [ ]:
# Lake budget summary table + signed exchange map for the final period:
display(model.packages.lak.results.q.budget.get(per=per))
model.packages.lak.results.map(field='q', per=per).plot()

## Observations: comparison tables and scatterplots

The bound target families compare measured vs simulated directly off the run.
(On this synthetic model the "measured" values are the model's own outputs,
so the scatter hugs the 1:1 line — with real field data this is your
calibration view.)


In [ ]:
display(model.targets.heads.compare().head(12))
display(model.targets.heads.stats())

# One-to-one calibration scatter (measured vs simulated heads):
model.targets.heads.calibration_plot()


## A real ensemble calibration (PESTPP-IES) — fast

`build_canonical_calibration_demo` perturbs the canonical model's K as a
starting point and samples head targets from the truth run. On the testing
profile a full prior→posterior IES cycle is ~20 forward runs. `style='grid'`
gives one geostatistically-correlated K parameter per Voronoi cell, and
`capture=True` records per-cell K fields so the ensemble is mappable.


In [ ]:
import shutil

from myflopy.modflow.mf6.canonical_calibration import build_canonical_calibration_demo

calib_root = root / 'calib'
shutil.rmtree(calib_root, ignore_errors=True)   # stale pest/ templates make PstFrom recurse

demo = build_canonical_calibration_demo(
    calib_root / 'model', config=mf.CanonicalModelConfig.testing(), n_head_wells=8
)

# Default workspace = <model workspace>/pest/fast_ies -- the front-door
# location `model.pest_runs` discovery looks in.
cal = demo.model.pest('fast_ies', start_datetime='2024-01-01')
cal.parameterize('k', style='grid', layers=[0], correlation=600.0,
                 bounds=(0.05, 20.0), physical=(0.001, 300.0), capture=True)
cal.parameterize('recharge', style='constant', bounds=(0.3, 3.0), physical=(0.0, 1e-2))
cal.observe(demo.head_targets)
cal.forecast(demo.forecast_targets)
cal.build('fast_ies.pst', noptmax=0)

t0 = time.time()
ies = cal.run_ies(reals=6, iterations=1, workers=4, lambda_scale_fac=1.0)
print(f'IES (prior + 1 iteration, 6 reals, 4 workers) in {time.time() - t0:.1f}s')


In [ ]:
# Ensemble diagnostics: phi convergence, phi distribution, ensemble vs observations
display(ies.plot_phi())
display(ies.plot_phi_distribution())
display(ies.plot_vs_obs())


In [ ]:
# Forecast uncertainty (prior vs posterior):
display(ies.forecasts())
display(ies.forecast(ies.forecast_names[0]).plot())


In [ ]:
# Parameter-field maps from the captured per-cell K: posterior mean,
# posterior spread, and how far the ensemble moved from the prior.
display(ies.plot_field('k', stat='mean', which='posterior', layer=0))
display(ies.plot_field('k', stat='std', layer=0))
display(ies.plot_field('k', stat='change', layer=0))
display(ies.field('k', layer=0).head())


In [ ]:
# Every calibration done on a model is discoverable and reopens for review:
for run in demo.model.pest_runs:
    print(run)
review = demo.model.pest_runs[0].review()
display(review.plot_phi())

report_path = ies.report(calib_root / 'fast_ies_report.html')
print('one-shot HTML report:', report_path)


## Where to go next

- `canonical_00` … `canonical_03`: the full-size model, packages/observations,
  visual diagnostics, PRT + parallel splits.
- `canonical_04` … `canonical_06`: PEST setup, calibration, and IES
  uncertainty on the validation profile.
- The suite re-runs everything on the bigger profile weekly in CI
  (`SIMPLE_MODFLOW_CANONICAL_PROFILE=validation pytest`), so the fast profile
  never drifts from the real thing.
